# zk-Crown · P4.5 · Fine-tuning attack on the attacker holdout

**Runtime > Change runtime type > T4 GPU before you start.**

The thief fine-tunes the stolen dual-watermarked `W*` (P3.6) on their own data, the 5,000-image
attacker holdout that `W*` never saw. They have no key, no triggers and no test set. The training
recipe is P0.5's (SGD with Nesterov momentum, weight decay 5e-4, augmentation, 1-epoch warmup,
cosine decay). Only the peak learning rate and the length change:

| peak LR | epochs | runs |
|---|---|---|
| 0.001 (gentle) | 5, 20, 60 | 3 |
| 0.01 | 5, 20, 60 | 3 |
| 0.05 (high) | 5, 20, 60 | 3 |
| 0.1 (aggressive: the peak LR `W*` was trained with) | 5, 20, 60 | 3 |

**This notebook never sees `K`.** It only produces the attacked weights. Accuracy and both
watermarks are scored later on the local machine with the key.

**Run order:** top to bottom (Runtime > Run all is fine). Cell 2 is the only cell you may need to
edit. Every check raises on failure, so Run all stops at the first problem.

**Expected runtime:** roughly 45 minutes in total (estimate, not a measurement): about 30 minutes
of fine-tuning plus tests, a smoke run and the CIFAR-10 download.

**Can be split across sessions.** Section 8 has one cell per learning rate. A run already complete
on Drive is skipped after its weights hash is checked, and a run cut off mid-way resumes from its
last epoch checkpoint.

**This notebook is a thin wrapper** (CLAUDE.md section 3). All logic is in `src/attacks/finetune.py`
and `experiments/run_attack_suite.py`.

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Configuration

In [ ]:
# ---- The only cell you may need to edit ------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/zk-crown"

# Where you put P4.5_colab.zip. If this file is not found, the next cell offers
# a manual upload instead.
ZIP_PATH = f"{DRIVE_ROOT}/handoff/P4.5_colab.zip"

# The dual-watermarked W* from P3.6 (results/p3.6_dual_wm_W_star.pt in the local repo).
SOURCE_PATH = f"{DRIVE_ROOT}/models/p3.6_dual_wm_W_star.pt"
# Its SHA-256. The harness refuses any other file. Do not edit.
EXPECTED_SOURCE_SHA256 = "7a9a9f141b55c7894b208c1f338f4c7385362f969b82b319ce5893fb4bb434c4"

# Attacked weights, apply records and per-epoch checkpoints go here.
OUT_DIR = f"{DRIVE_ROOT}/p4.5"

WORK_DIR = "/content/zk-crown"
# Outside WORK_DIR, which is wiped on every unpack, so a re-run after a
# disconnect in the same runtime does not download CIFAR-10 again.
DATA_ROOT = "/content/data"
# The smoke run writes to local disk, not Drive: it is thrown away.
SMOKE_OUT_DIR = "/content/p4.5_smoke"
SEED = 1337              # do not change between sessions
NUM_WORKERS = 2
LEARNING_RATES = ["0.001", "0.01", "0.05", "0.1"]   # the fixed grid; do not change

## 3. What GPU did we get?

Section 2.1: a T4 is expected, not guaranteed. If this says "no GPU", fix the runtime type before
going further.

In [ ]:
!nvidia-smi

## 4. Check the source model, unpack the handoff zip

In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# If Drive is not mounted, /content/drive/MyDrive/... would be created on local
# disk and every checkpoint would silently vanish with the session.
if DRIVE_ROOT.startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive is not mounted. Run section 1 first.")

# Fail now, not after the tests, if the source model is missing or wrong.
if not Path(SOURCE_PATH).is_file():
    raise FileNotFoundError(
        f"source model not found at {SOURCE_PATH}. Upload results/p3.6_dual_wm_W_star.pt "
        "from your local repo to that Drive path (see P4.5_INSTRUCTIONS.md)."
    )
source_sha256 = hashlib.sha256(Path(SOURCE_PATH).read_bytes()).hexdigest()
if source_sha256 != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(f"{SOURCE_PATH} has SHA-256 {source_sha256}, expected {EXPECTED_SOURCE_SHA256}")
print("source model found, hash verified:", SOURCE_PATH)

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)          # always unpack fresh; Drive holds the state that matters
work.mkdir(parents=True)

zip_path = Path(ZIP_PATH)
if not zip_path.is_file():
    print(f"{zip_path} not found. Upload P4.5_colab.zip now:")
    from google.colab import files

    uploaded = files.upload()
    name = next(iter(uploaded))
    zip_path = Path("/content") / name

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(work)

if not (work / "src").is_dir():
    inner = [d for d in work.iterdir() if d.is_dir() and (d / "src").is_dir()]
    if not inner:
        raise RuntimeError(f"no src/ found under {work}")
    work = inner[0]

REPO_DIR = str(work)
os.chdir(REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print(sorted(p.name for p in Path(REPO_DIR).iterdir()))
build_info = Path(REPO_DIR, "BUILD_INFO.json")
print(build_info.read_text() if build_info.is_file() else "no BUILD_INFO.json in the zip")


def run(args):
    """Run a command, stream its output into the cell, raise if it fails.

    `!cmd` does neither reliably: a non-zero exit does not fail the cell, so
    Run all would carry on past a failed test suite into the real run.
    """
    print("$", *args, flush=True)
    print(flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}; see output above")

## 5. Dependencies

`torch` and `torchvision` are skipped: Colab ships builds matched to its CUDA driver, and
installing over them tends to fail later as a CUDA error inside training rather than here.

In [ ]:
PREINSTALLED = {"torch", "torchvision"}

wanted = []
for line in Path("requirements.txt").read_text().splitlines():
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    if line.split("==")[0].split(">=")[0].strip().lower() in PREINSTALLED:
        continue
    wanted.append(line)

# P4.5 only fine-tunes. ezkl, streamlit, onnx and cryptography belong to later
# phases and pulling them in here costs minutes of setup for nothing.
NEEDED_FOR_P4_5 = {"numpy", "matplotlib", "pytest"}
wanted = [w for w in wanted if w.split("==")[0].strip().lower() in NEEDED_FOR_P4_5]
print("installing:", wanted)
if wanted:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *wanted], check=True)
print("done")

## 6. Fast checks before spending GPU time

The unit tests, then all 12 fine-tuning configs on the GPU with synthetic data and an untrained
stand-in model, written to local disk (`SMOKE_OUT_DIR`) and thrown away. Both raise on failure, so
Run all stops here. If either fails, report the traceback. Do not force section 8.

Some tests skip here because they need files that stay on your machine (CIFAR-10 under the repo's
`data/`, the trained weights, the key). That is expected.

In [ ]:
run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:warnings"])

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then run all again.")

run([
    sys.executable, "experiments/run_attack_suite.py", "apply",
    "--config", *[f"experiments/configs/p4.5_finetune_lr{lr}.json" for lr in LEARNING_RATES],
    "--task", "P4.5", "--smoke", "--device", "cuda", "--num-workers", 0,
    "--out-dir", SMOKE_OUT_DIR, "--seed", SEED,
])
print("\nSMOKE RUN done. Its numbers are meaningless and its files are not to be brought back.")

## 7. Download CIFAR-10 (the attacker holdout is carved out of the training set)

In [ ]:
from torchvision.datasets import CIFAR10

CIFAR10(DATA_ROOT, train=True, download=True)
print("CIFAR-10 training set ready at", DATA_ROOT)

## 8. The real fine-tuning runs, one cell per learning rate

Each cell runs the 5-, 20- and 60-epoch fine-tunes for one peak LR, starting from `W*` every
time. The harness checks the source hash first. Every run checkpoints each epoch to
`{OUT_DIR}/checkpoints/<run>/`, and when it finishes it writes the attacked weights
`{OUT_DIR}/p4.5_finetune_holdout_lr<LR>_<epochs>__seed1337.pt` and an apply record
`{OUT_DIR}/p4.5_apply_finetune_holdout_lr<LR>_<epochs>__seed1337__<time>.json`.

**Re-running any cell is safe.** A run that is already complete, with its weights hash verified, is
skipped. A run that was cut off resumes from its last epoch. Do not change `SEED` between attempts:
resuming under different settings is refused.

**What to watch.** The log prints `train_loss`, `train_acc` and `eval_acc` per epoch. `eval_acc` here
is accuracy **on the holdout the attacker is training on**, so it is a fit diagnostic and will
climb high, especially over 60 epochs. The high-LR runs may start with a visible accuracy dip in
the first epochs. That is part of what is being measured, not a fault: let them finish. Stop and
report only if the loss becomes `nan` or `eval_acc` sits at exactly `10.00%` for several epochs.

In [ ]:
def finetune(lr):
    run([
        sys.executable, "experiments/run_attack_suite.py", "apply",
        "--config", f"experiments/configs/p4.5_finetune_lr{lr}.json",
        "--task", "P4.5",
        "--source", "dual", "--source-path", SOURCE_PATH,
        "--device", "cuda", "--num-workers", NUM_WORKERS,
        "--data-root", DATA_ROOT,
        "--out-dir", OUT_DIR,
        "--seed", SEED,
    ])

In [ ]:
finetune("0.001")

In [ ]:
finetune("0.01")

In [ ]:
finetune("0.05")

In [ ]:
finetune("0.1")

## 9. Collect what comes back

For each of the 12 runs, finds its apply record on Drive and checks the weights file against the
hash in the record. A run is final only when both exist and match. Anything else means re-running
that learning rate's cell in section 8.

In [ ]:
import json

EPOCHS = [5, 20, 60]
incomplete = []
for lr in LEARNING_RATES:
    for epochs in EPOCHS:
        label = f"finetune_holdout_lr{lr}_{epochs}"
        records = sorted(Path(OUT_DIR).glob(f"p4.5_apply_{label}__seed{SEED}__*.json"))
        if not records:
            print(f"{label}: NO APPLY RECORD YET")
            incomplete.append(label)
            continue
        record = json.loads(records[-1].read_text())
        m, p = record["metrics"], record["params"]
        weights = Path(OUT_DIR, m["weights_file"])
        ok = weights.is_file() and hashlib.sha256(weights.read_bytes()).hexdigest() == m["weights_sha256"]
        if p["source"]["weights_sha256"] != EXPECTED_SOURCE_SHA256 or p["smoke"] or p["key_used"]:
            raise RuntimeError(f"{label}: record is not a real run on the expected source")
        if not ok:
            incomplete.append(label)
        info = m["info"]
        print(f"{label}: {'COMPLETE' if ok else 'WEIGHTS MISSING OR HASH MISMATCH'}")
        print(f"    apply record       : {records[-1].name}")
        print(f"    weights            : {m['weights_file']}  sha256 {m['weights_sha256']}  matches: {ok}")
        print(f"    first epoch        : train_loss {info['first_epoch']['train_loss']:.4f}  holdout_acc {info['first_epoch']['holdout_accuracy'] * 100:.2f}%")
        print(f"    last epoch         : train_loss {info['last_epoch']['train_loss']:.4f}  holdout_acc {info['last_epoch']['holdout_accuracy'] * 100:.2f}%")
        print(f"    epoch time         : {info['epoch_seconds_total'] / 60:.1f} min, summed over sessions   gpu {p['gpu']}")
        print(f"    code commit        : {record['git']['commit']} (dirty: {record['git']['dirty']})")

print()
if incomplete:
    print("*** NOT DONE:", ", ".join(incomplete), "-- re-run the matching cells in section 8. ***")
else:
    print("All 12 runs complete. Bring back the files listed in section 10.")

## 10. Bring back to the repo

From `{OUT_DIR}` (`MyDrive/zk-crown/p4.5/`), for **each** of the 12 runs:

1. The **newest** `p4.5_apply_finetune_holdout_lr<LR>_<epochs>__seed1337__*.json` (the one section 9
   printed) → the repo's `results/attacks/p4.5_apply/`.
2. `p4.5_finetune_holdout_lr<LR>_<epochs>__seed1337.pt` (about 1.2 MB each) → the same folder,
   next to the JSONs. `*.pt` is gitignored. Keep the Drive copies.

That is 24 files. Do **not** bring back `checkpoints/`, and nothing from the smoke run.

Then report back with the section 9 printout. Accuracy and both watermarks are scored locally with
the key; nothing is recorded before that. Finally run the cell below to flush Drive before the
runtime goes away.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
print("Drive flushed and unmounted. Remount (section 1) before running anything else.")